# Day 7 · Exercise 2: Chunk Text With Overlap

**What you'll build:** `chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]` — a function that splits a long string into fixed-size overlapping substrings using a sliding cursor.

**Why it matters:** Chunking is the first step in every long-document pipeline: without it, text that exceeds the model's context window either triggers an error or silently drops content.

## Your Implementation

In [ ]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into fixed-size chunks with a shared overlap region.

    Args:
        text:       The document to split.
        chunk_size: Maximum number of characters per chunk.
        overlap:    Number of characters shared between consecutive chunks.
                    Must be strictly less than chunk_size.

    Returns:
        A list of non-empty strings.  The last chunk may be shorter than
        chunk_size.  Returns an empty list when text is empty.

    Example:
        >>> chunk_text("ABCDEFGHIJ", chunk_size=6, overlap=2)
        ['ABCDEF', 'EFGHIJ']
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and is callable
    try:
        assert callable(chunk_text), 'chunk_text is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: empty input returns empty list
    try:
        result = chunk_text('', chunk_size=10, overlap=2)
        assert result == [], f'expected [], got {result!r}'
        print(f'{_PASS} Check 2/{total}: empty string returns []')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: text shorter than chunk_size produces exactly one chunk
    try:
        result = chunk_text('Hello', chunk_size=100, overlap=10)
        assert isinstance(result, list), f'expected list, got {type(result).__name__}'
        assert len(result) == 1, f'expected 1 chunk, got {len(result)}'
        assert result[0] == 'Hello', f'expected \'Hello\', got {result[0]!r}'
        print(f'{_PASS} Check 3/{total}: short text produces exactly one chunk containing the full text')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: consecutive chunks share the expected overlap region
    try:
        text = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'  # 26 chars
        # chunk_size=10, overlap=3 → step=7
        # chunk 0: chars 0-9  = 'ABCDEFGHIJ'
        # chunk 1: chars 7-16 = 'HIJKLMNOPQ'  (shares 'HIJ' = last 3 of chunk 0)
        result = chunk_text(text, chunk_size=10, overlap=3)
        assert len(result) >= 2, f'expected at least 2 chunks, got {len(result)}'
        tail_of_first = result[0][-3:]   # last `overlap` chars of chunk 0
        head_of_second = result[1][:3]   # first `overlap` chars of chunk 1
        assert tail_of_first == head_of_second, (
            f'overlap mismatch: chunk 0 tail={tail_of_first!r}, '
            f'chunk 1 head={head_of_second!r}'
        )
        print(f'{_PASS} Check 4/{total}: consecutive chunks share the overlap region ("{tail_of_first}")')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: overlap >= chunk_size raises ValueError
    try:
        raised = False
        try:
            chunk_text('some text', chunk_size=5, overlap=5)
        except ValueError:
            raised = True
        assert raised, 'expected ValueError when overlap >= chunk_size'
        print(f'{_PASS} Check 5/{total}: overlap >= chunk_size raises ValueError')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Hard character cuts can still split a word in two even with overlap. Try extending `chunk_text` into a `chunk_text_word_safe` variant that, after computing each slice, trims the right edge back to the nearest whitespace (so no chunk ever ends mid-word).

This foreshadows Day 9, where you will implement sentence-aware chunking — splitting on sentence boundaries instead of character counts for even cleaner results.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into fixed-size chunks with a shared overlap region."""
    if not text:
        return []
    if overlap >= chunk_size:
        raise ValueError(
            f"overlap ({overlap}) must be less than chunk_size ({chunk_size})"
        )

    step = chunk_size - overlap
    chunks = []
    cursor = 0

    while cursor < len(text):
        chunk = text[cursor : cursor + chunk_size]
        chunks.append(chunk)
        cursor += step

    return chunks
```

**Why this works:** The cursor advances by `step` (= `chunk_size - overlap`) rather than by the full `chunk_size`, so the last `overlap` characters of each chunk are repeated at the start of the next one — bridging any meaning that a hard cut would have lost. Python's slice syntax handles the end-of-string edge case automatically: `text[cursor : cursor + chunk_size]` never raises, it simply returns whatever characters remain, which is why the final chunk can be shorter than `chunk_size` with no special-case code required.
</details>